# Agent Metrics Visualization
This notebook connects to the local `metrics.db` SQLite database to visualize the agent's performance, token usage, and costs.

In [ ]:
import os
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

# Set plotting style
sns.set_theme(style="whitegrid")

# The db is located in the `runs` directory at the project root
db_path = os.path.join("..", "runs", "metrics.db")

try:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM metrics", conn)
    conn.close()

    if not df.empty:
        # Convert timestamp to datetime
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp")
        display(df.head())
    else:
        print("The metrics table is currently empty.")
except Exception as e:
    print(f"Error loading database: {e}\nMake sure you have run the agent at least once!")

## 1. Run Cost Over Time
Track how much each agent run costs over time.

In [ ]:
if "df" in locals() and not df.empty:
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=df, x="timestamp", y="cost", marker="o", color="purple")
    plt.title("Agent Run Cost Over Time")
    plt.xlabel("Run Timestamp")
    plt.ylabel("Cost ($)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 2. Tokens Consumed vs. Total Steps
See the relationship between the number of steps taken, the tokens consumed, and the duration.

In [ ]:
if "df" in locals() and not df.empty:
    plt.figure(figsize=(10, 6))
    # Scatter plot where size indicates duration and color indicates status
    sns.scatterplot(
        data=df,
        x="total_steps",
        y="total_tokens",
        hue="status",
        size="duration_seconds",
        sizes=(50, 400),
        alpha=0.7,
        palette="deep",
    )

    plt.title("Tokens Consumed vs. Total Steps (Size = Duration)")
    plt.xlabel("Total Steps")
    plt.ylabel("Total Tokens")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()